# NB28 · Testing, Debugging & Logging

`advanced/` · notebook 28 of 32 · 27 questions + a mini-project

Writing code is half the job. The other half is knowing it works, finding
out why it doesn't, and leaving a trail you can follow later. This notebook
covers all three: tests (`assert`, `unittest`, `doctest`, `unittest.mock`),
debugging (tracebacks, the `traceback` module, `breakpoint()`), and
`logging` and `warnings`.

A few house rules make the answers repeatable inside a notebook:

- Tests are run with `unittest.TextTestRunner(stream=io.StringIO())`. The
  runner's report goes into a string you ignore, and you print only stable
  facts: how many tests ran, whether they passed, which ones failed. (The
  report contains a timing line, and timings change every run.)
- Logs go to `sys.stdout` (`logging.basicConfig(stream=sys.stdout, force=True)`)
  or to a `StringIO` you print. By default logging writes to stderr, which
  Jupyter shows in red and which isn't part of the expected output.
- Warnings are caught with `warnings.catch_warnings()`, so you can inspect
  them instead of hoping they appear.

pytest is the tool most teams use day to day. It isn't in the standard
library, so the exercises use plain test functions in pytest *style* and
explain what pytest adds.

## Prerequisites

Before you start, you should be comfortable with:

- [ ] NB23: decorators
- [ ] NB24: context managers

## What you'll practise

- assert-based tests; arrange-act-assert; choosing edge cases
- unittest: TestCase, assertions, setUp, running inside a notebook
- pytest style: plain test functions, fixtures and parametrize (concepts)
- doctest
- unittest.mock: Mock, patch, side_effect, assert_called_with
- Debugging: reading tracebacks, the traceback module, breakpoint()/pdb
- logging: levels, getLogger, handlers, formatters; warnings

## How to use this notebook

1. Read the question and write your code in the cell beneath it.
2. Run the cell, then open **Expected output** and compare.
3. Stuck? Open **Hint** first. Open **Solution** only after a real attempt.
4. Once it works, open **Senior dev solution** to see how an experienced developer would write it, and why.

The **Solution** only uses what this notebook and the ones before it teach. The **Senior dev solution** sometimes uses later topics on purpose, and says which notebook covers them.

| Type | What you do |
|---|---|
| **Write** | Write code that prints the answer |
| **Predict** | Read the code and predict exactly what it prints |
| **Error** | Predict which error the code raises |
| **Fix** | The code has a bug. Find it and fix it |
| **Refactor** | The code works. Make it cleaner without changing its output |

| Level | Meaning |
|---|---|
| L1 | Extremely Easy: one idea, a line or two |
| L2 | Easy: one idea with a small twist |
| L3 | Intermediate: two or three ideas combined |
| L4 | Somewhat Difficult: needs care to get exactly right |
| L5 | Hard: several ideas and a trap or two |

---

## L1 · Extremely Easy

### NB28-Q01 · L1 · Assert Tests · Write

`add_vat()` adds 20% VAT to a net price. Write two `assert` statements
that check `add_vat(100)` is `120.0` and `add_vat(0)` is `0.0`. If both
pass, print:

```text
add_vat: 2 checks passed
```

**Starting code (already in the cell below):**

```python
def add_vat(net):
    return round(net * 1.2, 2)
```

<details><summary><b>Hint</b></summary>

`assert condition` does nothing when the condition is true and raises `AssertionError` when it's false. The print only runs if no assert failed.

</details>

<details><summary><b>Expected output</b></summary>

```text
add_vat: 2 checks passed
```

</details>

<details><summary><b>Solution</b></summary>

```python
assert add_vat(100) == 120.0
assert add_vat(0) == 0.0
print("add_vat: 2 checks passed")
```

An `assert` is the smallest possible test: a claim about your code that Python checks. If a claim is false, the program stops right there with `AssertionError`, so reaching the `print` proves every check above it passed. Testing `0` as well as a normal value is your first edge case.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
assert add_vat(100) == 120.0, add_vat(100)
assert add_vat(0) == 0.0, add_vat(0)
print("add_vat: 2 checks passed")
```

The part after the comma is the failure message. Putting the *actual* value there means a failing assert tells you what it got, not only that it was wrong. pytest does this for you automatically by rewriting asserts.

</details>

In [ ]:
def add_vat(net):
    return round(net * 1.2, 2)

# your code here


### NB28-Q02 · L1 · Logging · Predict

Logging is set to the `WARNING` level. Predict which lines appear.

**What does this print?**

```python
import logging
import sys

logging.basicConfig(stream=sys.stdout, level=logging.WARNING, force=True,
                    format="%(levelname)s %(message)s")
logging.debug("cache hit for user 7")
logging.info("order 1042 received")
logging.warning("disk 91% full")
logging.error("payment gateway timed out")
```

<details><summary><b>Hint</b></summary>

The five levels in order are DEBUG, INFO, WARNING, ERROR, CRITICAL. A level lets through messages at that level *and above*.

</details>

<details><summary><b>Expected output</b></summary>

```text
WARNING disk 91% full
ERROR payment gateway timed out
```

</details>

<details><summary><b>Solution</b></summary>

`level=logging.WARNING` is a threshold. `debug` and `info` are below it, so they are dropped. `warning` and `error` pass. This is the point of levels: you leave the debug calls in the code and turn them on by lowering the level, instead of adding and deleting `print()` calls. `force=True` replaces any logging setup an earlier cell made.

</details>

<details><summary><b>Senior dev solution</b></summary>

WARNING is also the default level when you don't configure anything, which is why a bare `logging.info(...)` seems to do nothing. Seniors configure logging once, at the program's entry point, and never inside library code.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB28-Q03 · L1 · Assert Tests · Error

A test for a discount function. Which error does this raise?

**Which error does this raise?**

```python
def apply_discount(price, percent):
    return price - price * percent

print(apply_discount(50, 10))
assert apply_discount(50, 10) == 45
```

<details><summary><b>Hint</b></summary>

Work out what `50 - 50 * 10` is. Is it 45?

</details>

<details><summary><b>Expected output</b></summary>

```text
-450
AssertionError
```

</details>

<details><summary><b>Solution</b></summary>

`AssertionError`. The function expects `percent` as a fraction (`0.1`), but the test passes `10`, so the result is `-450`. The test did its job: it caught a disagreement between how the function works and how people call it. Either the call or the function must change, and the test makes you decide which.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def apply_discount(price, percent):
    return price * (100 - percent) / 100

print(apply_discount(50, 10))
assert apply_discount(50, 10) == 45
```

The fix here takes percent as `10` meaning 10%, which matches how humans say it. A name like `percent` promises that; a fraction would be called `rate`. Good names stop this bug before any test runs.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

---

## L2 · Easy

### NB28-Q04 · L2 · Assert Tests · Write

`slugify()` turns a blog title into a URL piece. Pin down its behaviour
with **five** `assert` statements, one per case below, then print
`slugify: 5 checks passed`.

1. `"Hello World"` becomes `"hello-world"`
2. extra spaces at the ends are ignored: `"  Python Tips  "` becomes `"python-tips"`
3. several spaces in a row give one dash: `"a   b"` becomes `"a-b"`
4. punctuation is dropped: `"What's new?"` becomes `"whats-new"`
5. an empty title gives an empty slug: `""` becomes `""`

**Starting code (already in the cell below):**

```python
def slugify(title):
    kept = ""
    for ch in title.lower():
        if ch.isalnum() or ch == " ":
            kept += ch
    return "-".join(kept.split())
```

<details><summary><b>Hint</b></summary>

One `assert slugify(...) == ...` per case. Pick the inputs exactly as listed.

</details>

<details><summary><b>Expected output</b></summary>

```text
slugify: 5 checks passed
```

</details>

<details><summary><b>Solution</b></summary>

```python
assert slugify("Hello World") == "hello-world"
assert slugify("  Python Tips  ") == "python-tips"
assert slugify("a   b") == "a-b"
assert slugify("What's new?") == "whats-new"
assert slugify("") == ""
print("slugify: 5 checks passed")
```

Each assert pins one behaviour, and together they form a specification. Cases 2, 3 and 5 are *edge cases*: the places bugs hide (boundaries, repeats, empty input). A good test set has one normal case and then hunts the edges. If someone later "simplifies" `slugify` and breaks one, the matching assert says exactly which promise broke.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
cases = {
    "Hello World": "hello-world",
    "  Python Tips  ": "python-tips",
    "a   b": "a-b",
    "What's new?": "whats-new",
    "": "",
}
for title, expected in cases.items():
    assert slugify(title) == expected, (title, slugify(title))
print(f"slugify: {len(cases)} checks passed")
```

A table of inputs and expected outputs is the same idea as pytest's `@pytest.mark.parametrize`. Adding a case is one line, and the failure message shows the input that broke.

</details>

In [ ]:
def slugify(title):
    kept = ""
    for ch in title.lower():
        if ch.isalnum() or ch == " ":
            kept += ch
    return "-".join(kept.split())

# your code here


### NB28-Q05 · L2 · unittest · Write

Write a `unittest.TestCase` called `TestTemperature` with two tests for
`to_fahrenheit()`: `0` gives `32.0`, and `100` gives `212.0`. Run it
with the starting runner lines and print:

```text
ran: 2
ok: True
```

**Starting code (already in the cell below):**

```python
import io
import unittest

def to_fahrenheit(celsius):
    return celsius * 9 / 5 + 32

# Write the TestTemperature class here, then run it like this:
# suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestTemperature)
# result = unittest.TextTestRunner(stream=io.StringIO()).run(suite)
```

<details><summary><b>Hint</b></summary>

Subclass `unittest.TestCase`. Each test is a method whose name starts with `test`, and checks with `self.assertEqual(actual, expected)`. The result object has `testsRun` and `wasSuccessful()`.

</details>

<details><summary><b>Expected output</b></summary>

```text
ran: 2
ok: True
```

</details>

<details><summary><b>Solution</b></summary>

```python
class TestTemperature(unittest.TestCase):
    def test_freezing(self):
        self.assertEqual(to_fahrenheit(0), 32.0)

    def test_boiling(self):
        self.assertEqual(to_fahrenheit(100), 212.0)

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestTemperature)
result = unittest.TextTestRunner(stream=io.StringIO()).run(suite)
print("ran:", result.testsRun)
print("ok:", result.wasSuccessful())
```

`unittest` finds every method starting with `test` and runs each in a fresh instance of the class, so tests can't leak state into each other. The loader builds a *suite* from the class, the runner runs it and returns a result. Outside notebooks you'd write `unittest.main()`, but that tries to read command-line arguments and exit the program, which a notebook can't do. The `StringIO` stream swallows the runner's report (and its timing line).

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class TestTemperature(unittest.TestCase):
    def test_freezing_point(self):
        self.assertEqual(to_fahrenheit(0), 32.0)

    def test_boiling_point(self):
        self.assertEqual(to_fahrenheit(100), 212.0)

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestTemperature)
result = unittest.TextTestRunner(stream=io.StringIO()).run(suite)
print("ran:", result.testsRun)
print("ok:", result.wasSuccessful())
```

Test names are documentation: when one fails, the name is the first thing you read. `test_freezing_point` says what's being promised. `assertEqual` beats `assertTrue(a == b)` because its failure message shows both values.

</details>

In [ ]:
import io
import unittest

def to_fahrenheit(celsius):
    return celsius * 9 / 5 + 32

# Write the TestTemperature class here, then run it like this:
# suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestTemperature)
# result = unittest.TextTestRunner(stream=io.StringIO()).run(suite)

# your code here


### NB28-Q06 · L2 · doctest · Predict

The examples in a docstring can be run as tests. One of these three
examples is wrong. Predict the output.

**What does this print?**

```python
import doctest

def average(numbers):
    """Return the mean of a list of numbers.

    >>> average([2, 4])
    3.0
    >>> average([1, 2, 3, 4])
    2.5
    >>> average([5])
    5
    """
    return sum(numbers) / len(numbers)

runner = doctest.DocTestRunner()
for test in doctest.DocTestFinder().find(average, "average"):
    outcome = runner.run(test, out=lambda report: None)
    print("failed:", outcome.failed, "of", outcome.attempted)
```

<details><summary><b>Hint</b></summary>

doctest compares the *text* Python would print at the `>>>` prompt with the text written under it. What does `5 / 1` print?

</details>

<details><summary><b>Expected output</b></summary>

```text
failed: 1 of 3
```

</details>

<details><summary><b>Solution</b></summary>

doctest runs each `>>>` line and compares what Python *would display* with the line below it, as text. `average([5])` is `5 / 1`, which is `5.0`, and `"5.0"` is not `"5"`, so that example fails. `out=lambda report: None` throws the failure report away so only the counts print. In a normal `.py` file you'd call `doctest.testmod()` and read the report.

</details>

<details><summary><b>Senior dev solution</b></summary>

doctest is best for keeping documentation honest: the examples readers copy are guaranteed to work. It's a poor fit for detailed testing, because exact text matching breaks on things like float formatting and dict order. Seniors use it for examples and unittest or pytest for the rest.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB28-Q07 · L2 · Mocking · Write

`notify()` sends a message through whatever `sender` it is given. Test
it without sending anything real: pass a `Mock` as the sender, call
`notify(sender, "ada", 3)`, then print how many times the mock was
called and the arguments of the call:

```text
calls: 1
args: ('ada', 'You have 3 new messages')
```

**Starting code (already in the cell below):**

```python
from unittest.mock import Mock

def notify(sender, user, count):
    sender.send(user, f"You have {count} new messages")
```

<details><summary><b>Hint</b></summary>

A `Mock` records every call made on it. `sender.send.call_count` counts them and `sender.send.call_args.args` holds the positional arguments of the last call.

</details>

<details><summary><b>Expected output</b></summary>

```text
calls: 1
args: ('ada', 'You have 3 new messages')
```

</details>

<details><summary><b>Solution</b></summary>

```python
sender = Mock()
notify(sender, "ada", 3)
print("calls:", sender.send.call_count)
print("args:", sender.send.call_args.args)
```

A `Mock` accepts any attribute and any call, and remembers what happened. `sender.send` springs into existence the first time it's used. Because `notify` receives its sender as an argument, the test can hand it a fake one. Passing collaborators in like this is called *dependency injection*, and it's what makes code easy to test.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
sender = Mock()
notify(sender, "ada", 3)
sender.send.assert_called_once_with("ada", "You have 3 new messages")
print("calls:", sender.send.call_count)
print("args:", sender.send.call_args.args)
```

`assert_called_once_with` checks the count and the arguments in one line and raises `AssertionError` with a clear diff if either is wrong. In a test, seniors assert rather than print.

</details>

In [ ]:
from unittest.mock import Mock

def notify(sender, user, count):
    sender.send(user, f"You have {count} new messages")

# your code here


### NB28-Q08 · L2 · Logging · Write

Set up logging to standard output with the format
`LEVEL [logger name] message` and level `INFO`. Then use a logger named
`"shop"` to log one info message and one warning:

```text
INFO [shop] order 1042 received
WARNING [shop] stock for SKU-7 is low
```

**Starting code (already in the cell below):**

```python
import logging
import sys
```

<details><summary><b>Hint</b></summary>

`basicConfig` takes `stream`, `level`, `format` and `force`. In the format string, `%(levelname)s`, `%(name)s` and `%(message)s` are filled in. `logging.getLogger("shop")` gives you a named logger.

</details>

<details><summary><b>Expected output</b></summary>

```text
INFO [shop] order 1042 received
WARNING [shop] stock for SKU-7 is low
```

</details>

<details><summary><b>Solution</b></summary>

```python
logging.basicConfig(stream=sys.stdout, level=logging.INFO, force=True,
                    format="%(levelname)s [%(name)s] %(message)s")
log = logging.getLogger("shop")
log.info("order 1042 received")
log.warning("stock for SKU-7 is low")
```

`getLogger("shop")` returns the logger with that name, creating it the first time. Calling it again anywhere in the program returns the *same* logger. The format uses old-style `%(...)s` placeholders that logging fills in for every record. Including `%(name)s` tells you which part of the program spoke.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
logging.basicConfig(stream=sys.stdout, level=logging.INFO, force=True,
                    format="%(levelname)s [%(name)s] %(message)s")
log = logging.getLogger("shop")
order_id, sku = 1042, "SKU-7"
log.info("order %s received", order_id)
log.warning("stock for %s is low", sku)
```

Seniors pass values as arguments (`log.info("order %s", order_id)`) instead of building an f-string. Logging only formats the text if the message is actually emitted, and log tools can group messages by their unchanging template. In real modules the name is `__name__`.

</details>

In [ ]:
import logging
import sys

# your code here


### NB28-Q09 · L2 · Tracebacks · Predict

The `traceback` module turns an exception into text you can store or
log. Predict the output.

**What does this print?**

```python
import traceback

def parse_qty(text):
    if not text.isdigit():
        raise ValueError(f"bad quantity: {text!r}")
    return int(text)

try:
    parse_qty("two")
except ValueError as exc:
    line = traceback.format_exception_only(exc)[-1]
    print(line.strip())
    print(type(exc).__name__, "|", exc)
```

<details><summary><b>Hint</b></summary>

`format_exception_only` gives just the last line of a traceback: the part that names the exception and its message.

</details>

<details><summary><b>Expected output</b></summary>

```text
ValueError: bad quantity: 'two'
ValueError | bad quantity: 'two'
```

</details>

<details><summary><b>Solution</b></summary>

The last line of every traceback is `ExceptionName: message`, and `format_exception_only` returns exactly that (as a list of lines ending in `\n`, hence `.strip()`). `str(exc)` is only the message, so the second line builds the same thing by hand. `{text!r}` puts quotes around the bad value, which shows you blanks or odd characters.

</details>

<details><summary><b>Senior dev solution</b></summary>

When reading a full traceback, start at the **bottom**: the last line says what went wrong, and the frame just above it says where. Then read upwards to see how the program got there. Seniors also put the bad value in the message, as here, so the log alone explains the failure.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

---

## L3 · Intermediate

### NB28-Q10 · L3 · unittest · Write

Write `TestCart` with a `setUp` method that builds a cart holding
`("tea", 2.5, 2)` and `("milk", 1.2, 1)`. Add three tests:

- `test_total`: the total is `6.2`
- `test_add_more`: adding `("bread", 3.0, 1)` makes the total `9.2`
- `test_fresh_cart_each_test`: the cart has exactly 2 lines

Run the suite and print:

```text
ran: 3
failures: 0
errors: 0
```

**Starting code (already in the cell below):**

```python
import io
import unittest

class Cart:
    def __init__(self):
        self.lines = []

    def add(self, name, price, qty=1):
        self.lines.append((name, price, qty))

    def total(self):
        return round(sum(price * qty for name, price, qty in self.lines), 2)
```

<details><summary><b>Hint</b></summary>

`setUp(self)` runs before **every** test; store the cart on `self.cart`. The result has `failures` and `errors` lists. Compare money with `assertAlmostEqual` if you are worried about float noise.

</details>

<details><summary><b>Expected output</b></summary>

```text
ran: 3
failures: 0
errors: 0
```

</details>

<details><summary><b>Solution</b></summary>

```python
class TestCart(unittest.TestCase):
    def setUp(self):
        self.cart = Cart()
        self.cart.add("tea", 2.5, 2)
        self.cart.add("milk", 1.2, 1)

    def test_total(self):
        self.assertEqual(self.cart.total(), 6.2)

    def test_add_more(self):
        self.cart.add("bread", 3.0, 1)
        self.assertEqual(self.cart.total(), 9.2)

    def test_fresh_cart_each_test(self):
        self.assertEqual(len(self.cart.lines), 2)

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestCart)
result = unittest.TextTestRunner(stream=io.StringIO()).run(suite)
print("ran:", result.testsRun)
print("failures:", len(result.failures))
print("errors:", len(result.errors))
```

Each test follows *arrange, act, assert*: `setUp` arranges a known cart, the test acts (adds bread, or nothing), then asserts. Because `setUp` runs before each test on a fresh instance, `test_add_more` adding bread can't affect `test_fresh_cart_each_test`, whatever order they run in. A *failure* is an assert that didn't hold; an *error* is any other exception inside a test.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class TestCart(unittest.TestCase):
    def setUp(self):
        self.cart = Cart()
        self.cart.add("tea", 2.5, 2)
        self.cart.add("milk", 1.2, 1)

    def test_total(self):
        self.assertAlmostEqual(self.cart.total(), 6.2)

    def test_add_more(self):
        self.cart.add("bread", 3.0, 1)
        self.assertAlmostEqual(self.cart.total(), 9.2)

    def test_fresh_cart_each_test(self):
        self.assertEqual(len(self.cart.lines), 2)

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestCart)
result = unittest.TextTestRunner(stream=io.StringIO()).run(suite)
print("ran:", result.testsRun)
print("failures:", len(result.failures))
print("errors:", len(result.errors))
```

`assertAlmostEqual` compares floats to 7 decimal places, so a total of `9.200000000000001` still passes. The cart rounds today, but a test shouldn't break because someone removes that `round`. Seniors test the promise, not the implementation.

</details>

In [ ]:
import io
import unittest

class Cart:
    def __init__(self):
        self.lines = []

    def add(self, name, price, qty=1):
        self.lines.append((name, price, qty))

    def total(self):
        return round(sum(price * qty for name, price, qty in self.lines), 2)

# your code here


### NB28-Q11 · L3 · Mocking · Write

`is_expired()` reads the real clock, so testing it by waiting would take
a minute. Use `unittest.mock.patch` to replace `time.time` with a fake
that returns a fixed time, and check a token issued at `1000` with a
60-second lifetime, 59 and 60 seconds later:

```text
after 59s: False
after 60s: True
```

**Starting code (already in the cell below):**

```python
import time
from unittest.mock import patch

def is_expired(issued_at, ttl_seconds):
    return time.time() - issued_at >= ttl_seconds
```

<details><summary><b>Hint</b></summary>

`with patch("time.time", return_value=1059):` replaces the function for the duration of the block. Call `is_expired(1000, 60)` inside it.

</details>

<details><summary><b>Expected output</b></summary>

```text
after 59s: False
after 60s: True
```

</details>

<details><summary><b>Solution</b></summary>

```python
with patch("time.time", return_value=1059):
    print("after 59s:", is_expired(1000, 60))
with patch("time.time", return_value=1060):
    print("after 60s:", is_expired(1000, 60))
```

`patch("time.time", ...)` swaps the `time` attribute on the `time` module for a `Mock` that returns your number, and puts the real one back when the `with` block ends (it's a context manager, NB24). `is_expired` looks up `time.time` at call time, so it sees the fake. Testing exactly 59 and 60 checks the boundary, where `>` versus `>=` bugs live.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def is_expired(issued_at, ttl_seconds, now=time.time):
    return now() - issued_at >= ttl_seconds

print("after 59s:", is_expired(1000, 60, now=lambda: 1059))
print("after 60s:", is_expired(1000, 60, now=lambda: 1060))
```

Patching works, but seniors often prefer to *inject* the clock as a parameter with the real one as the default. Tests pass a lambda, no patching is needed, and the dependency on time is visible in the signature.

</details>

In [ ]:
import time
from unittest.mock import patch

def is_expired(issued_at, ttl_seconds):
    return time.time() - issued_at >= ttl_seconds

# your code here


### NB28-Q12 · L3 · Mocking · Write

`fetch_with_retry()` calls `fetch()` up to 3 times, retrying on
`ConnectionError`. Use a `Mock` with `side_effect` so the first two calls
fail and the third returns `{"temp": 21}`. Print the result and the number
of calls. Then make a second mock that **always** fails, and print the
message shown:

```text
result: {'temp': 21}
calls: 3
gave up after 3 calls
```

**Starting code (already in the cell below):**

```python
from unittest.mock import Mock

def fetch_with_retry(fetch, attempts=3):
    for attempt in range(1, attempts + 1):
        try:
            return fetch()
        except ConnectionError:
            if attempt == attempts:
                raise
```

<details><summary><b>Hint</b></summary>

`side_effect` can be a list: each call uses the next item, raising it if it's an exception and returning it otherwise. It can also be a single exception class, raised on every call.

</details>

<details><summary><b>Expected output</b></summary>

```text
result: {'temp': 21}
calls: 3
gave up after 3 calls
```

</details>

<details><summary><b>Solution</b></summary>

```python
flaky = Mock(side_effect=[ConnectionError("down"), ConnectionError("down"),
                           {"temp": 21}])
print("result:", fetch_with_retry(flaky))
print("calls:", flaky.call_count)

broken = Mock(side_effect=ConnectionError)
try:
    fetch_with_retry(broken)
except ConnectionError:
    print("gave up after", broken.call_count, "calls")
```

`return_value` gives the same answer every time; `side_effect` lets a mock *behave*. With a list, call 1 raises, call 2 raises, call 3 returns the dict. This is how you test failure paths that are hard to cause for real, like a network dropping exactly twice. The second mock proves the function gives up and re-raises instead of looping forever.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
flaky = Mock(side_effect=[ConnectionError("down")] * 2 + [{"temp": 21}])
result = fetch_with_retry(flaky)
print("result:", result)
print("calls:", flaky.call_count)

broken = Mock(side_effect=ConnectionError)
try:
    fetch_with_retry(broken)
except ConnectionError:
    print("gave up after", broken.call_count, "calls")
```

`[error] * 2 + [value]` makes the failure count obvious and easy to change. A list `side_effect` that runs out raises `StopIteration`, which is a quick sign the code made more calls than you expected (see Q28).

</details>

In [ ]:
from unittest.mock import Mock

def fetch_with_retry(fetch, attempts=3):
    for attempt in range(1, attempts + 1):
        try:
            return fetch()
        except ConnectionError:
            if attempt == attempts:
                raise

# your code here


### NB28-Q13 · L3 · Logging · Write

Build logging by hand instead of with `basicConfig`. Give the `"payments"`
logger its own `StreamHandler` writing into `buffer`, with a `Formatter`
of `%(levelname)s:%(name)s:%(message)s`. Set the logger to `DEBUG` and
stop it passing records up to the root logger. Log the three messages
below, then print what the buffer holds:

```text
DEBUG:payments:card check started
INFO:payments:charged 19.99
ERROR:payments:refund failed
```

**Starting code (already in the cell below):**

```python
import io
import logging

buffer = io.StringIO()
log = logging.getLogger("payments")
log.handlers.clear()   # so re-running this cell does not add a second handler
```

<details><summary><b>Hint</b></summary>

Create `logging.StreamHandler(buffer)`, call its `setFormatter(...)`, then `log.addHandler(...)`. `log.setLevel(logging.DEBUG)` and `log.propagate = False`. Finish with `print(buffer.getvalue(), end="")`.

</details>

<details><summary><b>Expected output</b></summary>

```text
DEBUG:payments:card check started
INFO:payments:charged 19.99
ERROR:payments:refund failed
```

</details>

<details><summary><b>Solution</b></summary>

```python
handler = logging.StreamHandler(buffer)
handler.setFormatter(logging.Formatter("%(levelname)s:%(name)s:%(message)s"))
log.addHandler(handler)
log.setLevel(logging.DEBUG)
log.propagate = False

log.debug("card check started")
log.info("charged 19.99")
log.error("refund failed")
print(buffer.getvalue(), end="")
```

Logging has three parts. A **logger** is what your code talks to, and it has a level. A **handler** decides where records go (a stream, a file, an email). A **formatter** decides what each line looks like. `basicConfig` wires these up on the root logger for you. `propagate = False` matters here: without it, every record would also go up to the root logger, and the handlers earlier cells installed there would print each line a second time.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
handler = logging.StreamHandler(buffer)
handler.setFormatter(logging.Formatter("%(levelname)s:%(name)s:%(message)s"))
log.addHandler(handler)
log.setLevel(logging.DEBUG)
log.propagate = False

amount = 19.99
log.debug("card check started")
log.info("charged %.2f", amount)
log.error("refund failed")
print(buffer.getvalue(), end="")
```

Writing logs to a `StringIO` is a standard way to *test* logging: you assert on the buffer. In an application, seniors configure handlers once (often with `logging.config.dictConfig`) and modules only call `getLogger(__name__)`.

</details>

In [ ]:
import io
import logging

buffer = io.StringIO()
log = logging.getLogger("payments")
log.handlers.clear()   # so re-running this cell does not add a second handler

# your code here


### NB28-Q14 · L3 · Warnings · Write

`old_total()` still works but is deprecated, so it issues a
`DeprecationWarning`. Call it inside `warnings.catch_warnings(record=True)`,
and print the result, how many warnings were caught, the warning's
category name and its message:

```text
total: 12
warnings: 1
DeprecationWarning: old_total() is deprecated, use total()
```

**Starting code (already in the cell below):**

```python
import warnings

def old_total(prices):
    warnings.warn("old_total() is deprecated, use total()",
                  DeprecationWarning, stacklevel=2)
    return sum(prices)
```

<details><summary><b>Hint</b></summary>

Inside the `with` block, call `warnings.simplefilter("always")` first so the warning isn't filtered out. The block gives you a list of caught warnings; each has `.category` and `.message`.

</details>

<details><summary><b>Expected output</b></summary>

```text
total: 12
warnings: 1
DeprecationWarning: old_total() is deprecated, use total()
```

</details>

<details><summary><b>Solution</b></summary>

```python
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    value = old_total([5, 7])

print("total:", value)
print("warnings:", len(caught))
print(f"{caught[0].category.__name__}: {caught[0].message}")
```

A warning is a message that does **not** stop the program, unlike an exception. Python filters warnings: by default `DeprecationWarning` is hidden outside `__main__`, and repeats are shown only once. `simplefilter("always")` turns that filtering off inside the block, and `record=True` collects the warnings into a list instead of printing them. Leaving the block restores the old filters.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    value = old_total([5, 7])

assert len(caught) == 1 and caught[0].category is DeprecationWarning
print("total:", value)
print("warnings:", len(caught))
print(f"{caught[0].category.__name__}: {caught[0].message}")
```

`stacklevel=2` makes the warning point at the *caller's* line, which is the line that needs changing. In unittest, `with self.assertWarns(DeprecationWarning):` does this check in one line. Python 3.13 also adds `@warnings.deprecated` for marking whole functions.

</details>

In [ ]:
import warnings

def old_total(prices):
    warnings.warn("old_total() is deprecated, use total()",
                  DeprecationWarning, stacklevel=2)
    return sum(prices)

# your code here


### NB28-Q15 · L3 · Assert Tests · Predict

These tests look fine, and `total()` has a bug. Predict the output.
(Jupyter may also show a warning in red. That's a clue.)

**What does this print?**

```python
def total(prices):
    return sum(prices[1:])

assert (total([2, 3]), 5)
assert total([]) == 0
print("all tests passed")
print(total([2, 3]))
```

<details><summary><b>Hint</b></summary>

What is the value of `(anything, 5)` when Python asks whether it's true?

</details>

<details><summary><b>Expected output</b></summary>

```text
all tests passed
3
```

</details>

<details><summary><b>Solution</b></summary>

The brackets turn the first assert into `assert <a tuple>`. A non-empty tuple is always truthy, so it passes no matter what `total` returns. The second test passes for the wrong reason too: an empty list is the one input where skipping the first item changes nothing. So the suite prints `all tests passed` while `total([2, 3])` is `3`. Python even notices and issues `SyntaxWarning: assertion is always true`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def total(prices):
    return sum(prices)

assert total([2, 3]) == 5, total([2, 3])
assert total([]) == 0
print("all tests passed")
print(total([2, 3]))
```

A test you have never seen fail proves nothing. Seniors break the code on purpose (or write the test first, before the fix) and watch the test go red. Test inputs should also be ones where a wrong answer would *look* different.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB28-Q16 · L3 · Tracebacks · Write

Loading the rows fails on bad data. Catch the `ValueError`, then use
`traceback.extract_tb()` on the exception's `__traceback__` to print the
chain of function names, and the exception's type:

```text
path: <module> -> load -> parse_row
error: ValueError
```

**Starting code (already in the cell below):**

```python
import traceback

def parse_row(row):
    name, qty = row.split(",")
    return name, int(qty)

def load(rows):
    result = []
    for row in rows:
        result.append(parse_row(row))
    return result

rows = ["apple,3", "pear,x"]
```

<details><summary><b>Hint</b></summary>

`extract_tb(tb)` returns a list of frame summaries, outermost first. Each has a `.name`. Join the names with `" -> "`.

</details>

<details><summary><b>Expected output</b></summary>

```text
path: <module> -> load -> parse_row
error: ValueError
```

</details>

<details><summary><b>Solution</b></summary>

```python
try:
    load(rows)
except ValueError as exc:
    frames = traceback.extract_tb(exc.__traceback__)
    names = [frame.name for frame in frames]
    print("path:", " -> ".join(names))
    print("error:", type(exc).__name__)
```

Every exception carries its traceback in `__traceback__`. `extract_tb` turns it into a list of frames from the place it was caught down to where it was raised: the cell itself (`<module>`), then `load`, then `parse_row`. `int()` is written in C, so it has no Python frame. This is the same information as the printed traceback, but as data you can filter, store or log.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
try:
    load(rows)
except ValueError as exc:
    names = [frame.name for frame in traceback.extract_tb(exc.__traceback__)]
    print("path:", " -> ".join(names))
    print("error:", type(exc).__name__)
```

Each frame summary also has `.filename`, `.lineno` and `.line`. In a real program seniors rarely build this by hand: `logging.exception("load failed")` inside an `except` block logs the message plus the full traceback.

</details>

In [ ]:
import traceback

def parse_row(row):
    name, qty = row.split(",")
    return name, int(qty)

def load(rows):
    result = []
    for row in rows:
        result.append(parse_row(row))
    return result

rows = ["apple,3", "pear,x"]

# your code here


### NB28-Q17 · L3 · Debugging · Fix

Someone debugged this function with `print()` and forgot to clean up.
The program should print only the line below. Remove the debug output,
and keep the details available as `DEBUG` log messages on a logger
named after the module (`__name__`) instead.

```text
Average: 4.0
```

**Buggy code (copied into the cell below):**

```python
def average_rating(ratings):
    print("DEBUG ratings =", ratings)
    total = sum(ratings)
    print("DEBUG total =", total)
    return total / len(ratings)

print(f"Average: {average_rating([4, 5, 3]):.1f}")
```

<details><summary><b>Hint</b></summary>

`log = logging.getLogger(__name__)`, then `log.debug(...)` in place of each debug print. Debug messages are hidden unless someone turns that level on.

</details>

<details><summary><b>Expected output</b></summary>

```text
Average: 4.0
```

</details>

<details><summary><b>Solution</b></summary>

```python
import logging

log = logging.getLogger(__name__)

def average_rating(ratings):
    log.debug("ratings = %s", ratings)
    total = sum(ratings)
    log.debug("total = %s", total)
    return total / len(ratings)

print(f"Average: {average_rating([4, 5, 3]):.1f}")
```

Leftover debug prints are a classic bug: they pollute output that other programs or people read, and they can leak data. `log.debug` keeps the information without the noise. The logger's level decides whether it appears, so next time you can turn it back on with one line of configuration instead of editing the function.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import logging

log = logging.getLogger(__name__)

def average_rating(ratings):
    total = sum(ratings)
    log.debug("ratings=%s total=%s", ratings, total)
    return total / len(ratings)

print(f"Average: {average_rating([4, 5, 3]):.1f}")
```

Seniors keep debug logs few and useful: one line with the key values rather than one per step. Many teams also add a linter rule (for example ruff's `T20`) that rejects stray `print()` calls in library code.

</details>

In [ ]:
def average_rating(ratings):
    print("DEBUG ratings =", ratings)
    total = sum(ratings)
    print("DEBUG total =", total)
    return total / len(ratings)

print(f"Average: {average_rating([4, 5, 3]):.1f}")


### NB28-Q18 · L3 · pytest Style · Write

pytest runs plain functions whose names start with `test_`, and gives
each one fresh data from a *fixture*. Imitate that by hand. Using the
fixture `make_basket()`, write three test functions:

- `test_count`: `count_items` of a fresh basket is `3`
- `test_in_stock`: `in_stock` of a fresh basket is `["apple"]`
- `test_fixture_is_fresh`: set `basket["pear"] = 5` on one basket, then
  check a **new** `make_basket()` still has `0` pears

Run them from a list, catching `AssertionError`, and print:

```text
PASS test_count
PASS test_in_stock
PASS test_fixture_is_fresh
3 passed, 0 failed
```

**Starting code (already in the cell below):**

```python
def make_basket():
    """Fixture: every test gets its own fresh basket."""
    return {"apple": 3, "pear": 0}

def count_items(basket):
    return sum(basket.values())

def in_stock(basket):
    return sorted(name for name, qty in basket.items() if qty > 0)
```

<details><summary><b>Hint</b></summary>

A function's name is in `func.__name__`. Loop over `[test_count, ...]`, call each inside `try`, and count passes and failures.

</details>

<details><summary><b>Expected output</b></summary>

```text
PASS test_count
PASS test_in_stock
PASS test_fixture_is_fresh
3 passed, 0 failed
```

</details>

<details><summary><b>Solution</b></summary>

```python
def test_count():
    assert count_items(make_basket()) == 3

def test_in_stock():
    assert in_stock(make_basket()) == ["apple"]

def test_fixture_is_fresh():
    first = make_basket()
    first["pear"] = 5
    assert make_basket()["pear"] == 0

passed = 0
failed = 0
for test in [test_count, test_in_stock, test_fixture_is_fresh]:
    try:
        test()
        print("PASS", test.__name__)
        passed += 1
    except AssertionError:
        print("FAIL", test.__name__)
        failed += 1
print(f"{passed} passed, {failed} failed")
```

This is pytest in miniature. Tests are plain functions using plain `assert`. A *fixture* is a function that builds fresh test data, so one test changing its basket can't break another (the third test proves it). The loop is the *runner*: it calls each test, and an `AssertionError` means that test failed while the others still run.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def test_count():
    assert count_items(make_basket()) == 3

def test_in_stock():
    assert in_stock(make_basket()) == ["apple"]

def test_fixture_is_fresh():
    first = make_basket()
    first["pear"] = 5
    assert make_basket()["pear"] == 0

tests = [obj for name, obj in list(globals().items())
         if name.startswith("test_") and callable(obj)]
outcomes = {}
for test in tests:
    try:
        test()
        outcomes[test.__name__] = "PASS"
    except AssertionError:
        outcomes[test.__name__] = "FAIL"
    except Exception:
        outcomes[test.__name__] = "ERROR"
for name, outcome in outcomes.items():
    print(outcome, name)
passed = list(outcomes.values()).count("PASS")
print(f"{passed} passed, {len(outcomes) - passed} failed")
```

The senior runner *discovers* tests by name, as pytest does, and tells a failed assert apart from a crash (`ERROR`). With pytest installed you'd write the fixture as `@pytest.fixture def basket(): ...` and the test as `def test_count(basket): ...`: pytest sees the argument name and passes in the fixture's result. `@pytest.mark.parametrize("title, slug", [...])` runs one test over a table of inputs. Run it with `pytest` in a terminal; it isn't used here because it's not part of the standard library.

</details>

In [ ]:
def make_basket():
    """Fixture: every test gets its own fresh basket."""
    return {"apple": 3, "pear": 0}

def count_items(basket):
    return sum(basket.values())

def in_stock(basket):
    return sorted(name for name, qty in basket.items() if qty > 0)

# your code here


---

## L4 · Somewhat Difficult

### NB28-Q19 · L4 · Assert Tests · Fix

This shopping cart has **three planted bugs**, and its only test passes.
First add checks (using `check`) for these three promises, and watch
them fail:

1. adding `"tea"` again adds to its quantity: `tea` x2 then `tea` x1 gives `3`
2. `total(10)` takes 10% off: 3 teas at `2.0` cost `5.4`
3. `count()` counts units, not lines: `3`

Then fix the cart so every check prints `PASS`:

```text
PASS total 6.0
PASS re-add adds qty 3
PASS 10% off 5.4
PASS count units 3
```

**Buggy code (copied into the cell below):**

```python
class Cart:
    def __init__(self):
        self.items = {}

    def add(self, name, price, qty=1):
        self.items[name] = (price, qty)

    def total(self, discount_percent=0):
        subtotal = sum(price * qty for price, qty in self.items.values())
        return round(subtotal - discount_percent / 100, 2)

    def count(self):
        return len(self.items)

def check(label, actual, expected):
    print("PASS" if actual == expected else "FAIL", label, actual)

cart = Cart()
cart.add("tea", 2.0, 3)
check("total", cart.total(), 6.0)
```

<details><summary><b>Hint</b></summary>

Build the cart for checks 1 to 3 the same way: `add("tea", 2.0, 2)` then `add("tea", 2.0, 1)`. Each bug is one line: look at what `add` does to an existing name, what the discount subtracts, and what `count` measures.

</details>

<details><summary><b>Expected output</b></summary>

```text
PASS total 6.0
PASS re-add adds qty 3
PASS 10% off 5.4
PASS count units 3
```

</details>

<details><summary><b>Solution</b></summary>

```python
class Cart:
    def __init__(self):
        self.items = {}

    def add(self, name, price, qty=1):
        old_qty = self.items.get(name, (price, 0))[1]
        self.items[name] = (price, old_qty + qty)

    def total(self, discount_percent=0):
        subtotal = sum(price * qty for price, qty in self.items.values())
        return round(subtotal * (100 - discount_percent) / 100, 2)

    def count(self):
        return sum(qty for price, qty in self.items.values())

def check(label, actual, expected):
    print("PASS" if actual == expected else "FAIL", label, actual)

cart = Cart()
cart.add("tea", 2.0, 3)
check("total", cart.total(), 6.0)

cart = Cart()
cart.add("tea", 2.0, 2)
cart.add("tea", 2.0, 1)
check("re-add adds qty", cart.items["tea"][1], 3)
check("10% off", cart.total(10), 5.4)
check("count units", cart.count(), 3)
```

The one original test used a single `add` with no discount, which is exactly the path that avoids all three bugs. That's why bugs survive: tests only cover what someone thought to check. The new checks target the promises (re-adding, discounting, counting), fail first, and pass after the fixes: `add` now adds to the old quantity, the discount scales the subtotal instead of subtracting `0.1`, and `count` sums quantities.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from collections import Counter

class Cart:
    def __init__(self):
        self.qty = Counter()
        self.price = {}

    def add(self, name, price, qty=1):
        self.price[name] = price
        self.qty[name] += qty

    def total(self, discount_percent=0):
        subtotal = sum(self.price[n] * q for n, q in self.qty.items())
        return round(subtotal * (100 - discount_percent) / 100, 2)

    def count(self):
        return self.qty.total()

def check(label, actual, expected):
    print("PASS" if actual == expected else "FAIL", label, actual)

cart = Cart()
cart.add("tea", 2.0, 3)
check("total", cart.total(), 6.0)

cart = Cart()
cart.add("tea", 2.0, 2)
cart.add("tea", 2.0, 1)
check("re-add adds qty", cart.qty["tea"], 3)
check("10% off", cart.total(10), 5.4)
check("count units", cart.count(), 3)
```

A `Counter` (NB15) makes "add to the quantity" the natural operation (`+=` on a missing key starts at 0), so bug 1 can't come back. `Counter.total()` sums the counts. In a real project these checks would be unittest or pytest tests that keep running on every change, so a fixed bug that comes back is caught immediately (a *regression test*).

</details>

In [ ]:
class Cart:
    def __init__(self):
        self.items = {}

    def add(self, name, price, qty=1):
        self.items[name] = (price, qty)

    def total(self, discount_percent=0):
        subtotal = sum(price * qty for price, qty in self.items.values())
        return round(subtotal - discount_percent / 100, 2)

    def count(self):
        return len(self.items)

def check(label, actual, expected):
    print("PASS" if actual == expected else "FAIL", label, actual)

cart = Cart()
cart.add("tea", 2.0, 3)
check("total", cart.total(), 6.0)


### NB28-Q20 · L4 · Mocking · Fix

The test patches `time.time`, but `session_age()` still sees the real
clock, so the check prints `False`. Fix the **patch target** (don't
change `session_age` or its import) so it prints:

```text
age is 300: True
```

**Buggy code (copied into the cell below):**

```python
from time import time
from unittest.mock import patch

def session_age(started_at):
    return time() - started_at

with patch("time.time", return_value=1300.0):
    print("age is 300:", session_age(1000.0) == 300.0)
```

<details><summary><b>Hint</b></summary>

`from time import time` copies the function into *this* module under the name `time`. Which module's `time` does `session_age` look up when it runs? This notebook's module is called `__main__`.

</details>

<details><summary><b>Expected output</b></summary>

```text
age is 300: True
```

</details>

<details><summary><b>Solution</b></summary>

```python
from time import time
from unittest.mock import patch

def session_age(started_at):
    return time() - started_at

with patch("__main__.time", return_value=1300.0):
    print("age is 300:", session_age(1000.0) == 300.0)
```

The rule is **patch where it's looked up, not where it's defined**. `from time import time` creates a second name, `__main__.time`, pointing at the real function. Patching `time.time` replaces the attribute on the `time` module, but `session_age` never looks there: it uses its own module's name. In a project this looks like `patch("billing.time")` for code in `billing.py` that did `from time import time`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import time

def session_age(started_at, now=time.time):
    return now() - started_at

print("age is 300:", session_age(1000.0, now=lambda: 1300.0) == 300.0)
```

Mocking the wrong path is one of the most common testing bugs, and it fails silently: the test runs against the real thing. Injecting the dependency (as in Q11's senior) avoids patch paths entirely. When you do patch, check it worked, e.g. by asserting the mock was called.

</details>

In [ ]:
from time import time
from unittest.mock import patch

def session_age(started_at):
    return time() - started_at

with patch("time.time", return_value=1300.0):
    print("age is 300:", session_age(1000.0) == 300.0)


### NB28-Q21 · L4 · unittest · Write

`parse_price()` turns shop text like `"£4.50"` into a float. Write a
`TestParsePrice` class with these five tests, run it, and print the
number of tests run and the **names** of the tests in `failures` and in
`errors`:

- `test_plain`: `"4.50"` gives `4.5`
- `test_pound_sign`: `"£4.50"` gives `4.5`
- `test_thousands`: `"£1,299.00"` gives `1299.0`
- `test_empty_raises`: `""` raises `ValueError`
- `test_negative_rejected`: `"-3"` raises `ValueError`

```text
ran: 5
failures: ['test_negative_rejected']
errors: ['test_thousands']
```

**Starting code (already in the cell below):**

```python
import io
import unittest

def parse_price(text):
    return float(text.strip().lstrip("£"))
```

<details><summary><b>Hint</b></summary>

`with self.assertRaises(ValueError): parse_price("")` passes only if the block raises. `result.failures` and `result.errors` are lists of `(test, traceback_text)` pairs, and `test.id()` ends with the method name.

</details>

<details><summary><b>Expected output</b></summary>

```text
ran: 5
failures: ['test_negative_rejected']
errors: ['test_thousands']
```

</details>

<details><summary><b>Solution</b></summary>

```python
class TestParsePrice(unittest.TestCase):
    def test_plain(self):
        self.assertEqual(parse_price("4.50"), 4.5)

    def test_pound_sign(self):
        self.assertEqual(parse_price("£4.50"), 4.5)

    def test_thousands(self):
        self.assertEqual(parse_price("£1,299.00"), 1299.0)

    def test_empty_raises(self):
        with self.assertRaises(ValueError):
            parse_price("")

    def test_negative_rejected(self):
        with self.assertRaises(ValueError):
            parse_price("-3")

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestParsePrice)
result = unittest.TextTestRunner(stream=io.StringIO()).run(suite)
print("ran:", result.testsRun)
print("failures:", [test.id().split(".")[-1] for test, text in result.failures])
print("errors:", [test.id().split(".")[-1] for test, text in result.errors])
```

The two kinds of red mean different things. `test_negative_rejected` is a **failure**: the check ran and didn't hold (`"-3"` parsed happily, nothing was raised). `test_thousands` is an **error**: `float("1,299.00")` raised a `ValueError` the test didn't expect, so the test crashed before reaching its assert. Failures usually mean wrong answers; errors often mean unhandled input. Both tell you exactly which promise to fix next.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
class TestParsePrice(unittest.TestCase):
    def test_plain(self):
        self.assertEqual(parse_price("4.50"), 4.5)

    def test_pound_sign(self):
        self.assertEqual(parse_price("£4.50"), 4.5)

    def test_thousands(self):
        self.assertEqual(parse_price("£1,299.00"), 1299.0)

    def test_empty_raises(self):
        with self.assertRaises(ValueError):
            parse_price("")

    def test_negative_rejected(self):
        with self.assertRaisesRegex(ValueError, "negative"):
            parse_price("-3")

def names(pairs):
    return sorted(test._testMethodName for test, _ in pairs)

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestParsePrice)
result = unittest.TextTestRunner(stream=io.StringIO()).run(suite)
print("ran:", result.testsRun)
print("failures:", names(result.failures))
print("errors:", names(result.errors))
```

`assertRaisesRegex` also checks the message, so the test can't pass because of some *other* `ValueError` (a test passing for the wrong reason). Sorting the names keeps the output stable however the runner orders tests.

</details>

In [ ]:
import io
import unittest

def parse_price(text):
    return float(text.strip().lstrip("£"))

# your code here


### NB28-Q22 · L4 · unittest · Refactor

Four test methods do the same thing with different numbers. Refactor
them into **one** test that loops over a table of cases, using
`self.subTest(...)` so each case is still reported separately. Keep the
printed output the same.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
import io
import unittest

def grade(score):
    if score >= 90:
        return "A"
    if score >= 70:
        return "B"
    if score >= 50:
        return "C"
    return "F"

class TestGrade(unittest.TestCase):
    def test_90(self):
        self.assertEqual(grade(90), "A")

    def test_89(self):
        self.assertEqual(grade(89), "B")

    def test_50(self):
        self.assertEqual(grade(50), "C")

    def test_49(self):
        self.assertEqual(grade(49), "F")

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestGrade)
result = unittest.TextTestRunner(stream=io.StringIO()).run(suite)
print("ok:", result.wasSuccessful())
print("failures:", len(result.failures))
```

<details><summary><b>Hint</b></summary>

Put the `(score, expected)` pairs in a list. Inside the loop, `with self.subTest(score=score):` wraps the assert.

</details>

<details><summary><b>Expected output</b></summary>

```text
ok: True
failures: 0
```

</details>

<details><summary><b>Solution</b></summary>

```python
import io
import unittest

def grade(score):
    if score >= 90:
        return "A"
    if score >= 70:
        return "B"
    if score >= 50:
        return "C"
    return "F"

class TestGrade(unittest.TestCase):
    def test_boundaries(self):
        cases = [(90, "A"), (89, "B"), (50, "C"), (49, "F")]
        for score, expected in cases:
            with self.subTest(score=score):
                self.assertEqual(grade(score), expected)

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestGrade)
result = unittest.TextTestRunner(stream=io.StringIO()).run(suite)
print("ok:", result.wasSuccessful())
print("failures:", len(result.failures))
```

A plain loop of asserts would stop at the first failing case and hide the rest. `subTest` records each case on its own, so if two boundaries broke you'd see both, labelled with their `score`. Adding a case is now one tuple, not a new method.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import io
import unittest

def grade(score):
    for cutoff, letter in [(90, "A"), (70, "B"), (50, "C")]:
        if score >= cutoff:
            return letter
    return "F"

class TestGrade(unittest.TestCase):
    CASES = [(90, "A"), (89, "B"), (70, "B"), (69, "C"), (50, "C"), (49, "F")]

    def test_boundaries(self):
        for score, expected in self.CASES:
            with self.subTest(score=score):
                self.assertEqual(grade(score), expected)

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestGrade)
result = unittest.TextTestRunner(stream=io.StringIO()).run(suite)
print("ok:", result.wasSuccessful())
print("failures:", len(result.failures))
```

Now that cases are cheap, the senior tests **both sides of every boundary** (90/89, 70/69, 50/49). The original missed the 70 boundary completely. This is pytest's `parametrize` idea in unittest form.

</details>

In [ ]:
import io
import unittest

def grade(score):
    if score >= 90:
        return "A"
    if score >= 70:
        return "B"
    if score >= 50:
        return "C"
    return "F"

class TestGrade(unittest.TestCase):
    def test_90(self):
        self.assertEqual(grade(90), "A")

    def test_89(self):
        self.assertEqual(grade(89), "B")

    def test_50(self):
        self.assertEqual(grade(50), "C")

    def test_49(self):
        self.assertEqual(grade(49), "F")

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestGrade)
result = unittest.TextTestRunner(stream=io.StringIO()).run(suite)
print("ok:", result.wasSuccessful())
print("failures:", len(result.failures))


### NB28-Q23 · L4 · Logging · Refactor

This function uses a `VERBOSE` flag and `print()` to report progress and
problems. Refactor it to use a logger, configured once with
`basicConfig` to write to `sys.stdout` with the format
`%(levelname)s %(message)s`. Keep the output exactly the same.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
VERBOSE = True

def process(orders):
    total = 0
    for order_id, amount in orders:
        if VERBOSE:
            print("INFO processing", order_id)
        if amount <= 0:
            print("WARNING skipping", order_id, "bad amount")
            continue
        total += amount
    return total

print("total:", process([("A1", 20), ("A2", -5), ("A3", 15)]))
```

<details><summary><b>Hint</b></summary>

`INFO` and `WARNING` are already log levels. The `VERBOSE` flag becomes the logger's level: `INFO` shows both, `WARNING` would hide the progress lines.

</details>

<details><summary><b>Expected output</b></summary>

```text
INFO processing A1
INFO processing A2
WARNING skipping A2 bad amount
INFO processing A3
total: 35
```

</details>

<details><summary><b>Solution</b></summary>

```python
import logging
import sys

logging.basicConfig(stream=sys.stdout, level=logging.INFO, force=True,
                    format="%(levelname)s %(message)s")
log = logging.getLogger("orders")

def process(orders):
    total = 0
    for order_id, amount in orders:
        log.info("processing %s", order_id)
        if amount <= 0:
            log.warning("skipping %s bad amount", order_id)
            continue
        total += amount
    return total

print("total:", process([("A1", 20), ("A2", -5), ("A3", 15)]))
```

The hand-made `VERBOSE` switch and the `"INFO"`/`"WARNING"` prefixes were a home-grown logging system. Logging already has levels, prefixes (the format) and an on/off switch (the level), so the function shrinks to what it actually does. The final `print` stays a `print`: it's the program's *result*, not a diagnostic.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import logging
import sys

log = logging.getLogger("orders")

def process(orders):
    total = 0
    for order_id, amount in orders:
        log.info("processing %s", order_id)
        if amount <= 0:
            log.warning("skipping %s bad amount", order_id)
            continue
        total += amount
    return total

if __name__ == "__main__":
    logging.basicConfig(stream=sys.stdout, level=logging.INFO, force=True,
                        format="%(levelname)s %(message)s")
    print("total:", process([("A1", 20), ("A2", -5), ("A3", 15)]))
```

Seniors keep the two kinds of output apart: results go to stdout, diagnostics go through logging (usually to stderr or a file). Configuring logging under `if __name__ == "__main__":` (NB31) means importing the module never changes the importer's logging setup.

</details>

In [ ]:
VERBOSE = True

def process(orders):
    total = 0
    for order_id, amount in orders:
        if VERBOSE:
            print("INFO processing", order_id)
        if amount <= 0:
            print("WARNING skipping", order_id, "bad amount")
            continue
        total += amount
    return total

print("total:", process([("A1", 20), ("A2", -5), ("A3", 15)]))


### NB28-Q24 · L4 · Mocking · Error

A `Mock` built with `spec=Mailer` stands in for the real mailer. The
second call has a typo. Which error does this raise?

**Which error does this raise?**

```python
from unittest.mock import Mock


class Mailer:
    def send(self, to, body):
        raise RuntimeError("real email!")


mailer = Mock(spec=Mailer)
mailer.send("ada@example.com", "Your order shipped")
mailer.sned("bob@example.com", "Your order shipped")
print("sent")
```

<details><summary><b>Hint</b></summary>

A plain `Mock` accepts *any* attribute. What does `spec=` change about that?

</details>

<details><summary><b>Expected output</b></summary>

```text
AttributeError
```

</details>

<details><summary><b>Solution</b></summary>

`AttributeError`. `spec=Mailer` makes the mock only accept attributes the real `Mailer` has. `send` exists, so the first call works; `sned` doesn't, so it fails loudly. Without `spec`, `mailer.sned(...)` would quietly succeed and your test would pass while testing nothing.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from unittest.mock import create_autospec


class Mailer:
    def send(self, to, body):
        raise RuntimeError("real email!")


mailer = create_autospec(Mailer, instance=True)
mailer.send("ada@example.com", "Your order shipped")
print(mailer.send.call_count)
```

`create_autospec` goes one step further than `spec`: it also checks the *arguments*. Calling `mailer.send("ada@example.com")` with one argument missing would raise `TypeError`, just like the real method. Seniors autospec by default so mocks can't drift away from the code they stand in for.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

---

## L5 · Hard

### NB28-Q25 · L5 · Logging · Write

Build a small logger hierarchy for a shop. Send everything to `log_output`
through **one** handler on the `shop` logger, formatted as
`LEVEL name: message`. Set the levels so that:

- `shop` logs `INFO` and above,
- `shop.cart` is more talkative and logs `DEBUG` and above,
- `shop.payment` is quieter and logs only `WARNING` and above.

Stop `shop` passing records on to the root logger, then run these lines
and print what was logged:

```python
logging.getLogger("shop").debug("startup details")
logging.getLogger("shop").info("shop open")
logging.getLogger("shop.cart").debug("cart has 3 items")
logging.getLogger("shop.payment").info("charging card")
logging.getLogger("shop.payment").error("card declined")
```

**Starting code (already in the cell below):**

```python
import io
import logging

log_output = io.StringIO()
```

<details><summary><b>Hint</b></summary>

Loggers named `shop.cart` and `shop.payment` are children of `shop`: their records flow up to `shop`'s handler. Each logger's own level decides whether a record is created at all.

</details>

<details><summary><b>Expected output</b></summary>

```text
INFO shop: shop open
DEBUG shop.cart: cart has 3 items
ERROR shop.payment: card declined
```

</details>

<details><summary><b>Solution</b></summary>

```python
shop = logging.getLogger("shop")
shop.handlers.clear()
handler = logging.StreamHandler(log_output)
handler.setFormatter(logging.Formatter("%(levelname)s %(name)s: %(message)s"))
shop.addHandler(handler)
shop.setLevel(logging.INFO)
shop.propagate = False
logging.getLogger("shop.cart").setLevel(logging.DEBUG)
logging.getLogger("shop.payment").setLevel(logging.WARNING)

logging.getLogger("shop").debug("startup details")
logging.getLogger("shop").info("shop open")
logging.getLogger("shop.cart").debug("cart has 3 items")
logging.getLogger("shop.payment").info("charging card")
logging.getLogger("shop.payment").error("card declined")
print(log_output.getvalue(), end="")
```

A record is checked against the level of the logger that *creates* it. `shop.cart` is set to `DEBUG`, so its debug line is created and then flows up to `shop`'s handler, even though `shop` itself is at `INFO`. `shop.payment` drops its `INFO` line but keeps the `ERROR`. `handlers.clear()` stops duplicate lines when you rerun the cell, and `propagate = False` stops a root handler (say from `basicConfig` in an earlier cell) printing everything twice.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import logging.config

logging.config.dictConfig({
    "version": 1,
    "disable_existing_loggers": False,
    "formatters": {"short": {"format": "%(levelname)s %(name)s: %(message)s"}},
    "handlers": {"memory": {"class": "logging.StreamHandler",
                            "stream": log_output, "formatter": "short"}},
    "loggers": {
        "shop": {"level": "INFO", "handlers": ["memory"], "propagate": False},
        "shop.cart": {"level": "DEBUG"},
        "shop.payment": {"level": "WARNING"},
    },
})

logging.getLogger("shop").debug("startup details")
logging.getLogger("shop").info("shop open")
logging.getLogger("shop.cart").debug("cart has 3 items")
logging.getLogger("shop.payment").info("charging card")
logging.getLogger("shop.payment").error("card declined")
print(log_output.getvalue(), end="")
```

Real applications describe logging as data with `dictConfig`, usually loaded from a config file, and configure it once at startup. Modules then only ever call `logging.getLogger(__name__)`, so the hierarchy follows the package layout automatically and nobody's module adds handlers.

</details>

In [ ]:
import io
import logging

log_output = io.StringIO()

# your code here


### NB28-Q26 · L5 · Mocking · Predict

Two small modules are built in memory: `pricing` defines `get_rate()`, and
`checkout` does `from pricing import get_rate`. Two patches try to make the
tax rate zero. Predict both lines. (This is the classic mocking mistake.)

**What does this print?**

```python
import sys
import types
from unittest.mock import patch

pricing = types.ModuleType("pricing")
exec("def get_rate():\n    return 0.20\n", pricing.__dict__)
sys.modules["pricing"] = pricing

checkout = types.ModuleType("checkout")
exec("from pricing import get_rate\n\n"
     "def total(amount):\n    return round(amount * (1 + get_rate()), 2)\n",
     checkout.__dict__)
sys.modules["checkout"] = checkout

with patch("pricing.get_rate", return_value=0.0):
    print(checkout.total(100))
with patch("checkout.get_rate", return_value=0.0):
    print(checkout.total(100))
```

<details><summary><b>Hint</b></summary>

`from pricing import get_rate` copies the name into `checkout` when the import runs. Which name does `total()` actually look up?

</details>

<details><summary><b>Expected output</b></summary>

```text
120.0
100.0
```

</details>

<details><summary><b>Solution</b></summary>

`patch` swaps a name *where it is looked up*, not where it was defined. `checkout` has its own name `get_rate`, copied in at import time, so patching `pricing.get_rate` changes nothing for `total()`: it prints `120.0`. Patching `checkout.get_rate` replaces the name `total()` really uses: `100.0`.

</details>

<details><summary><b>Senior dev solution</b></summary>

The rule seniors repeat: *patch where it's used, not where it's defined*. If `checkout` had done `import pricing` and called `pricing.get_rate()`, then patching `pricing.get_rate` would work. That's one reason many teams prefer `import module` over `from module import name`.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB28-Q27 · L5 · pytest Style · Write

Write `check_cases(func, cases)`, a tiny version of pytest's
`@pytest.mark.parametrize`. For each `(argument, expected)` pair it calls
`func(argument)` and prints one line, then a summary. A case that raises an
exception counts as an error, not a crash:

```text
PASS slug('Hello World')
PASS slug('  Python   Rocks ')
FAIL slug('Déjà Vu'): expected 'deja-vu', got 'déjà-vu'
PASS slug('')
ERROR slug(None): AttributeError
3/5 passed
```

Then call `check_cases(slug, cases)`.

**Starting code (already in the cell below):**

```python
def slug(text):
    return "-".join(text.lower().split())


cases = [
    ("Hello World", "hello-world"),
    ("  Python   Rocks ", "python-rocks"),
    ("Déjà Vu", "deja-vu"),
    ("", ""),
    (None, ""),
]
```

<details><summary><b>Hint</b></summary>

Build the call text with `func.__name__` and `{argument!r}`. Wrap only the call in `try` so a raising case prints `ERROR` and the loop carries on.

</details>

<details><summary><b>Expected output</b></summary>

```text
PASS slug('Hello World')
PASS slug('  Python   Rocks ')
FAIL slug('Déjà Vu'): expected 'deja-vu', got 'déjà-vu'
PASS slug('')
ERROR slug(None): AttributeError
3/5 passed
```

</details>

<details><summary><b>Solution</b></summary>

```python
def check_cases(func, cases):
    passed = 0
    for argument, expected in cases:
        call = f"{func.__name__}({argument!r})"
        try:
            got = func(argument)
        except Exception as exc:
            print(f"ERROR {call}: {type(exc).__name__}")
            continue
        if got == expected:
            passed += 1
            print(f"PASS {call}")
        else:
            print(f"FAIL {call}: expected {expected!r}, got {got!r}")
    print(f"{passed}/{len(cases)} passed")


check_cases(slug, cases)
```

Each case is independent: one failure or exception never stops the others, which is exactly what a test runner promises. `!r` shows strings with quotes, so `''` and `' '` look different in the report. The `try` only wraps the call, so a bug in the checker itself still crashes loudly instead of being reported as a test error.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from dataclasses import dataclass


@dataclass
class Outcome:
    call: str
    status: str
    detail: str = ""


def run_case(func, argument, expected):
    call = f"{func.__name__}({argument!r})"
    try:
        got = func(argument)
    except Exception as exc:
        return Outcome(call, "ERROR", type(exc).__name__)
    if got == expected:
        return Outcome(call, "PASS")
    return Outcome(call, "FAIL", f"expected {expected!r}, got {got!r}")


def check_cases(func, cases):
    outcomes = [run_case(func, argument, expected) for argument, expected in cases]
    for outcome in outcomes:
        detail = f": {outcome.detail}" if outcome.detail else ""
        print(f"{outcome.status} {outcome.call}{detail}")
    passed = sum(outcome.status == "PASS" for outcome in outcomes)
    print(f"{passed}/{len(cases)} passed")


check_cases(slug, cases)
```

The senior version separates *running* a case from *reporting* it. Once outcomes are data (a dataclass, NB21), you can also write them to JSON, count them by status or rerun only the failures. That split is how pytest itself is built. In a real project you'd just write `@pytest.mark.parametrize("text, expected", cases)` and let pytest do it.

</details>

In [ ]:
def slug(text):
    return "-".join(text.lower().split())


cases = [
    ("Hello World", "hello-world"),
    ("  Python   Rocks ", "python-rocks"),
    ("Déjà Vu", "deja-vu"),
    ("", ""),
    (None, ""),
]

# your code here


---

## Mini-project

Pull together everything from this notebook.

## NB28-P · L5 · Mini-project: Bank Test Suite · Write

The banking module below has **three bugs**. Your job is to prove them with
tests, fix them, and prove the fixes.

1. Write a `unittest.TestCase` with these five tests, built so the same
   tests can run against any account class and transfer function:
   - `test_deposit_adds`: depositing 5 into 10 returns 15
   - `test_deposit_rejects_negative`: `deposit(-5)` raises `ValueError`
   - `test_withdraw_exact_balance`: withdrawing all 10 of 10 returns 0
   - `test_withdraw_too_much`: withdrawing 11 of 10 raises `InsufficientFunds`
   - `test_transfer_is_all_or_nothing`: a transfer of 50 from an account
     holding 5 raises `InsufficientFunds` **and** leaves both balances unchanged
2. Run them against the buggy code and report.
3. Write `FixedAccount` (a subclass of `Account`) and `fixed_transfer`, and
   run the same tests again.

Report with this helper (copy it in), printing failures in sorted order:

```python
def report(label, test_case):
    suite = unittest.TestLoader().loadTestsFromTestCase(test_case)
    result = unittest.TextTestRunner(stream=io.StringIO()).run(suite)
    failed = sorted(test.id().split(".")[-1] for test, _ in result.failures + result.errors)
    print(f"{label}: {result.testsRun} tests, {len(failed)} failed")
    for name in failed:
        print("  FAIL", name)
```

```text
Buggy version: 5 tests, 3 failed
  FAIL test_deposit_rejects_negative
  FAIL test_transfer_is_all_or_nothing
  FAIL test_withdraw_exact_balance
Fixed version: 5 tests, 0 failed
```

**Starting code (already in the cell below):**

```python
import io
import unittest


class InsufficientFunds(Exception):
    pass


class Account:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

    def deposit(self, amount):
        self.balance += amount
        return self.balance

    def withdraw(self, amount):
        if amount >= self.balance:
            raise InsufficientFunds(self.owner)
        self.balance -= amount
        return self.balance


def transfer(source, target, amount):
    target.deposit(amount)
    source.withdraw(amount)
```

<details><summary><b>Hint</b></summary>

Write a function `make_tests(account_cls, transfer_fn)` that defines and returns the TestCase class, so the tests can be pointed at either version. The three bugs: a missing check, a `>=` that should be `>`, and two lines in the wrong order.

</details>

<details><summary><b>Expected output</b></summary>

```text
Buggy version: 5 tests, 3 failed
  FAIL test_deposit_rejects_negative
  FAIL test_transfer_is_all_or_nothing
  FAIL test_withdraw_exact_balance
Fixed version: 5 tests, 0 failed
```

</details>

<details><summary><b>Solution</b></summary>

```python
def report(label, test_case):
    suite = unittest.TestLoader().loadTestsFromTestCase(test_case)
    result = unittest.TextTestRunner(stream=io.StringIO()).run(suite)
    failed = sorted(test.id().split(".")[-1] for test, _ in result.failures + result.errors)
    print(f"{label}: {result.testsRun} tests, {len(failed)} failed")
    for name in failed:
        print("  FAIL", name)


def make_tests(account_cls, transfer_fn):
    class AccountTests(unittest.TestCase):
        def test_deposit_adds(self):
            self.assertEqual(account_cls("Ada", 10).deposit(5), 15)

        def test_deposit_rejects_negative(self):
            with self.assertRaises(ValueError):
                account_cls("Ada", 10).deposit(-5)

        def test_withdraw_exact_balance(self):
            self.assertEqual(account_cls("Ada", 10).withdraw(10), 0)

        def test_withdraw_too_much(self):
            with self.assertRaises(InsufficientFunds):
                account_cls("Ada", 10).withdraw(11)

        def test_transfer_is_all_or_nothing(self):
            source, target = account_cls("Ada", 5), account_cls("Bob", 0)
            with self.assertRaises(InsufficientFunds):
                transfer_fn(source, target, 50)
            self.assertEqual((source.balance, target.balance), (5, 0))

    return AccountTests


report("Buggy version", make_tests(Account, transfer))

class FixedAccount(Account):
    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("deposit must be positive")
        return super().deposit(amount)

    def withdraw(self, amount):
        if amount > self.balance:
            raise InsufficientFunds(self.owner)
        self.balance -= amount
        return self.balance


def fixed_transfer(source, target, amount):
    source.withdraw(amount)
    target.deposit(amount)


report("Fixed version", make_tests(FixedAccount, fixed_transfer))
```

Each bug shows up as one failing test. `deposit(-5)` doesn't raise, `withdraw(10)` wrongly raises because of `>=`, and `transfer` credits the target *before* checking the source, so a failed transfer still creates money. Writing the tests first, watching them fail, then fixing until they pass is the red-green loop of test-driven development. Running the *same* tests against both versions proves the fix and not just a new test.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def report(label, test_case):
    suite = unittest.TestLoader().loadTestsFromTestCase(test_case)
    result = unittest.TextTestRunner(stream=io.StringIO()).run(suite)
    failed = sorted(test.id().split(".")[-1] for test, _ in result.failures + result.errors)
    print(f"{label}: {result.testsRun} tests, {len(failed)} failed")
    for name in failed:
        print("  FAIL", name)


class AccountContract:
    """Tests any account class and transfer function. Subclasses pick which."""

    account_cls = None
    transfer_fn = None

    def account(self, balance):
        return self.account_cls("Ada", balance)

    def test_deposit_adds(self):
        self.assertEqual(self.account(10).deposit(5), 15)

    def test_deposit_rejects_negative(self):
        with self.assertRaises(ValueError):
            self.account(10).deposit(-5)

    def test_withdraw_exact_balance(self):
        self.assertEqual(self.account(10).withdraw(10), 0)

    def test_withdraw_too_much(self):
        with self.assertRaises(InsufficientFunds):
            self.account(10).withdraw(11)

    def test_transfer_is_all_or_nothing(self):
        source, target = self.account(5), self.account(0)
        with self.assertRaises(InsufficientFunds):
            self.transfer_fn(source, target, 50)
        self.assertEqual((source.balance, target.balance), (5, 0))

class FixedAccount(Account):
    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("deposit must be positive")
        return super().deposit(amount)

    def withdraw(self, amount):
        if amount > self.balance:
            raise InsufficientFunds(self.owner)
        self.balance -= amount
        return self.balance


def fixed_transfer(source, target, amount):
    source.withdraw(amount)
    target.deposit(amount)


class BuggyTests(AccountContract, unittest.TestCase):
    account_cls = Account
    transfer_fn = staticmethod(transfer)


class FixedTests(AccountContract, unittest.TestCase):
    account_cls = FixedAccount
    transfer_fn = staticmethod(fixed_transfer)


report("Buggy version", BuggyTests)
report("Fixed version", FixedTests)
```

A *contract test* mixin is how seniors test several implementations of one interface: the tests live once in `AccountContract`, and each subclass just names the implementation. `staticmethod(...)` stops Python turning the stored function into a method. In pytest you'd write the same idea as a parametrized fixture. Also note what the fix *didn't* need: no test had to change, which is the sign the tests describe behaviour, not implementation.

</details>

In [ ]:
import io
import unittest


class InsufficientFunds(Exception):
    pass


class Account:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

    def deposit(self, amount):
        self.balance += amount
        return self.balance

    def withdraw(self, amount):
        if amount >= self.balance:
            raise InsufficientFunds(self.owner)
        self.balance -= amount
        return self.balance


def transfer(source, target, amount):
    target.deposit(amount)
    source.withdraw(amount)

# your code here


---

## Done

Next up: **NB29 · Performance Memory** in `advanced/`.